# Day 15 Evaluation Notebook
**Author:** Sheshikala  
**Topic:** Graph RAG + Advanced RAG Techniques Evaluation

This notebook evaluates the Day 15 components:
- Graph RAG multi-hop query accuracy
- Advanced RAG technique comparison
- Recall@1 and Recall@3 metrics
- Week 3 summary

In [ ]:
# Setup
import sys
import math
import re
print('Day 15 Evaluation Notebook')
print('=' * 40)

## Part 1: Graph RAG Stats

In [ ]:
from graph_rag import build_ml_knowledge_graph, GraphRAG

graph = build_ml_knowledge_graph()
rag = GraphRAG(graph)

stats = rag.get_graph_stats()
print('Graph Statistics:')
print(f'  Total nodes : {stats["total_nodes"]}')
print(f'  Total edges : {stats["total_edges"]}')
print(f'  Node types  : {stats["node_types"]}')
print(f'  Edge types  : {stats["relation_types"]}')

## Part 2: Multi-Hop Query Results

In [ ]:
print('Multi-Hop Query: Researcher -> Experiments -> Metrics')
print('-' * 50)
for researcher in ['Ananya', 'Vikram', 'Priya', 'Rohan']:
    exps = rag.get_researcher_experiments(researcher)
    print(f'\n{researcher}: {len(exps)} experiments')
    for e in exps:
        print(f'  {e["exp_id"]} | {e["model"]} | {e["metrics"]}')

In [ ]:
print('Multi-Hop Query: Best Metric -> Experiment -> Researcher')
print('-' * 50)
for metric in ['accuracy', 'f1', 'mse', 'bleu']:
    result = rag.get_best_experiment_by_metric(metric)
    if result:
        print(f'Best {metric}: {result["value"]} | {result["exp_id"]} | {result["model"]} | by {result["researcher"]}')

## Part 3: Advanced RAG Technique Comparison

In [ ]:
from advanced_rag import (
    KNOWLEDGE_BASE, retrieve,
    HyDERetriever, QueryExpansionRetriever,
    RerankingRetriever, MultiQueryRetriever,
    ContextualCompressionRetriever
)

test_query = 'Which researcher achieved the best accuracy?'
print(f'Query: {test_query}')
print('-' * 50)

baseline = retrieve(test_query, KNOWLEDGE_BASE, top_k=1)
print(f'Baseline top result:')
print(f'  {baseline[0]["text"][:100]}...')
print(f'  Score: {baseline[0]["score"]}')

In [ ]:
techniques = [
    ('HyDE',                HyDERetriever(KNOWLEDGE_BASE)),
    ('Query Expansion',     QueryExpansionRetriever(KNOWLEDGE_BASE)),
    ('Reranking',           RerankingRetriever(KNOWLEDGE_BASE)),
    ('Multi-Query',         MultiQueryRetriever(KNOWLEDGE_BASE)),
    ('Compression',         ContextualCompressionRetriever(KNOWLEDGE_BASE)),
]

for name, retriever in techniques:
    result = retriever.retrieve(test_query, top_k=1)
    if isinstance(result, dict):
        top = result.get('results', [{}])[0]
    else:
        top = result[0] if result else {}
    text = top.get('compressed', top.get('text', ''))[:80]
    print(f'[{name}]: {text}...')

## Part 4: Recall@K Evaluation

In [ ]:
# Recall@K: does the correct document appear in top-K results?

eval_set = [
    {'query': 'Which experiment used BERT?',           'expected_id': 'K01'},
    {'query': 'Who used RoBERTa?',                     'expected_id': 'K02'},
    {'query': 'ResNet50 experiment results',           'expected_id': 'K03'},
    {'query': 'LSTM time series forecasting MSE',      'expected_id': 'K04'},
    {'query': 'GPT-2 fine-tuning BLEU score',          'expected_id': 'K05'},
    {'query': 'DistilBERT vs BERT speed comparison',   'expected_id': 'K06'},
    {'query': 'NLP-Corpus-v2 dataset properties',      'expected_id': 'K11'},
]

def recall_at_k(eval_set, docs, k):
    hits = 0
    for case in eval_set:
        results = retrieve(case['query'], docs, top_k=k)
        ids = [r['id'] for r in results]
        if case['expected_id'] in ids:
            hits += 1
    return round(hits / len(eval_set), 3)

r1 = recall_at_k(eval_set, KNOWLEDGE_BASE, k=1)
r3 = recall_at_k(eval_set, KNOWLEDGE_BASE, k=3)
r5 = recall_at_k(eval_set, KNOWLEDGE_BASE, k=5)

print('Recall@K Results (Baseline Retriever)')
print('-' * 40)
print(f'Recall@1 : {r1}')
print(f'Recall@3 : {r3}')
print(f'Recall@5 : {r5}')

In [ ]:
# Per-query breakdown
print('Per-query Recall@3 breakdown:')
print('-' * 50)
for case in eval_set:
    results = retrieve(case['query'], KNOWLEDGE_BASE, top_k=3)
    ids = [r['id'] for r in results]
    hit = case['expected_id'] in ids
    status = 'HIT ' if hit else 'MISS'
    print(f'[{status}] {case["query"][:50]}')
    print(f'         Expected: {case["expected_id"]} | Got: {ids}')

## Part 5: Week 3 Summary

In [ ]:
print('WEEK 3 SUMMARY')
print('=' * 50)

summary = [
    ('Day 11', 'PostgreSQL',              'Schema design, complex queries, JSONB, indexing, partitioning'),
    ('Day 12', 'MongoDB + Redis',         'Document store, aggregation pipelines, caching, Celery tasks'),
    ('Day 13', 'Vector DB',               'ChromaDB, FAISS, hybrid search BM25+semantic, Pinecone interface'),
    ('Day 14', 'RAG Pipeline',            'Chunking, embeddings, hybrid retrieval, QA app, Groq integration'),
    ('Day 15', 'Graph RAG + Advanced RAG','Knowledge graph, HyDE, reranking, multi-query, streaming QA'),
]

for day, topic, what_i_built in summary:
    print(f'\n{day}: {topic}')
    print(f'  Built: {what_i_built}')

print('\nKey Takeaway:')
print('  Vector search + Graph traversal + Advanced retrieval')
print('  together cover most real-world RAG use cases.')
print('  Next: production deployment, evaluation frameworks.')